In [ ]:
import pandas as pd 
import os 
import pydicom
from PIL import Image, ImageFilter
from modules.cardiac import *
from modules.implant import *
from modules.comp import *
from modules.helpers import *
from modules.small import *

# use pandarallel instead of pandas for large dataset
# from pandarallel import pandarallel
# pandarallel.initialize(progress_bar=True)

In [ ]:
image_level_df = pd.read_csv('sample_dataset.csv')

In [7]:
image_level_df

,file_name,cardiac_label,spot_label,stand_label,half_label,implant_label,laterality
0,sample_1.png,1,0,0,0,0,R
1,sample_2.png,0,0,0,0,1,R
2,sample_3.png,0,0,1,0,0,L
3,sample_4.png,0,0,1,0,0,L
4,sample_5.png,0,1,0,0,0,R
5,sample_6.png,0,0,0,1,0,R


In [ ]:
# clahe for cardiac device detection 
clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8))
SOURCE = "sample_data/images"

def check_all(row):
    filename = row['file_name']

    if '.dcm' in filename:
        fov_flip = row['FieldOfViewHorizontalFlip']
        laterality = row['ImageLaterality']
        img = dicom_to_8bit_pil(SOURCE, filename) 
        laterality = flip_lat(laterality, fov_flip)
    
    else: # image file not DICOM
        laterality = row['laterality']
        img = Image.open(f"{SOURCE}/{filename}")


    is_paddle = detect_compression_paddle(
        img, 
        laterality=laterality, 
        top_percent=5,
        debug=False, 
        img_name=filename,
    )

    has_small = detect_small_breast_compression_paddle(img)

    is_implant, *_ = detect_implant_in_image(np.array(img), 
                                             laterality,
                                             top_percent=30)

    # blurring before cardiac device method 
    img_blurred = img.filter(ImageFilter.GaussianBlur(radius=15))
    is_cardiac = detect_cardiac_device_in_image(
        img_blurred, clahe, laterality,
        min_area=60000,
        max_area=500000,
        circularity_min=0.25,
        circularity_max=0.65,
        aspect_ratio_max=4.5,
        top_percent=1,
        density_min=0.5, 
        density_max=1.0,
        intensity_threshold=225, 
        debug=False,
        morph_kernel_size=25
    )[0]

    return pd.Series({
        'pred_has_stand_paddle':  int(is_paddle),
        'pred_has_half':     int(has_small),
        'pred_has_implant': int(is_implant),
        'pred_has_cardiac': int(is_cardiac),
    })

In [9]:
cols = ['pred_has_paddle', 'pred_has_sbp', 'pred_has_implant', 'pred_has_cardiac']

image_level_df[cols] = (
    image_level_df.apply(check_all, axis=1)
)

inside first not catch
inside first not catch
